In [1]:
import pandas as pd

In [2]:
#from google.colab import drive
#drive.mount('/content/drive')

In [3]:
df = pd.read_csv("../data/spam.csv", encoding="latin-1")

In [4]:
df.shape

(5572, 5)

In [5]:
df.columns

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')

In [6]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [7]:
df = df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4' ])

In [8]:
df.shape

(5572, 2)

In [9]:
#rename columns
df = df.rename(columns={'v1': 'label' , 'v2': 'message'})

In [10]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [11]:
#find missing values
df.isnull().sum()

label      0
message    0
dtype: int64

In [12]:
df["label"].unique()

array(['ham', 'spam'], dtype=object)

In [13]:
#show duplicate values
df.duplicated().sum()

np.int64(403)

In [14]:
#drop duplicate rows
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [15]:
df.describe()

,label,message
count,5169,5169
unique,2,5169
top,ham,Rofl. Its true to its name
freq,4516,1


In [16]:
#show distribution
df['label'].value_counts()

label
ham     4516
spam     653
Name: count, dtype: int64

In [17]:
df["label"].value_counts(normalize=True) * 100

label
ham     87.366996
spam    12.633004
Name: proportion, dtype: float64

**Text Preprocessing**

In [18]:
#remove punctuation and converting message to lower case
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_message"] = df["message"].apply(clean_text)

In [19]:
df["clean_message"] = df["message"].apply(clean_text)

In [20]:
df[["message", "clean_message"]].head(10)

,message,clean_message
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,"Nah I don't think he goes to usf, he lives aro...",nah i don t think he goes to usf he lives arou...
5,FreeMsg Hey there darling it's been 3 week's n...,freemsg hey there darling it s been 3 week s n...
6,Even my brother is not like to speak with me. ...,even my brother is not like to speak with me t...
7,As per your request 'Melle Melle (Oru Minnamin...,as per your request melle melle oru minnaminun...
8,WINNER!! As a valued network customer you have...,winner as a valued network customer you have b...
9,Had your mobile 11 months or more? U R entitle...,had your mobile 11 months or more u r entitled...


In [21]:
#check empty message
df["clean_message"].str.strip().eq("").sum()

np.int64(2)

In [22]:
df[df["clean_message"].str.strip().eq("")]

,label,message,clean_message
3374,ham,:),
4822,ham,:-) :-),


In [23]:
df = df[~df["clean_message"].str.strip().eq("")].copy()

In [24]:
df["clean_message"].str.strip().eq("").sum()

np.int64(0)

In [25]:
df.reset_index(drop=True, inplace=True)

In [26]:
df = df.drop_duplicates().copy()

In [27]:
df.reset_index(drop=True, inplace=True)

In [28]:
#final check on data
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

print("\nLabels:")
print(df["label"].value_counts())

print("\nEmpty cleaned messages:",
      df["clean_message"].str.strip().eq("").sum())

Shape: (5167, 3)

Missing values:
label            0
message          0
clean_message    0
dtype: int64

Duplicates: 0

Labels:
label
ham     4514
spam     653
Name: count, dtype: int64

Empty cleaned messages: 0


**Train and Test Split**

In [29]:
from sklearn.model_selection import train_test_split

X = df["clean_message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [30]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training samples: 4133
Testing samples: 1034

Training distribution:
label
ham     87.369949
spam    12.630051
Name: proportion, dtype: float64

Testing distribution:
label
ham     87.330754
spam    12.669246
Name: proportion, dtype: float64


In [31]:
df.to_csv("../data/cleaned_spam.csv", index=False)